# NNDL_Proj1

In [1]:
from pandas import read_csv
from keras.models import Sequential
from keras.layers.convolutional import Conv1D
from keras.layers.convolutional import MaxPooling1D
from keras.layers import Flatten
from keras.layers import Dense
from keras.wrappers.scikit_learn import KerasRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
# calculate accuracy measures and confusion matrix
from sklearn import metrics
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
%matplotlib inline

Using TensorFlow backend.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&scope=email%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdocs.test%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.photos.readonly%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fpeopleapi.readonly&response_type=code

Enter your authorization code:
··········
Mounted at /content/drive


1. Read the dataset

In [0]:
df  = pd.read_csv('/content/drive/My Drive/GL_AIML/Residency_6/Projects/bank.csv') 

In [4]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
df.shape

(10000, 14)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
RowNumber          10000 non-null int64
CustomerId         10000 non-null int64
Surname            10000 non-null object
CreditScore        10000 non-null int64
Geography          10000 non-null object
Gender             10000 non-null object
Age                10000 non-null int64
Tenure             10000 non-null int64
Balance            10000 non-null float64
NumOfProducts      10000 non-null int64
HasCrCard          10000 non-null int64
IsActiveMember     10000 non-null int64
EstimatedSalary    10000 non-null float64
Exited             10000 non-null int64
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


2. Drop the columns which are unique for all users like IDs (5 points)

In [0]:
df = df.drop(['RowNumber'], axis=1)
df = df.drop(['CustomerId'], axis=1)
df= df.drop(['Surname'], axis=1)


In [8]:
df.shape

(10000, 11)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
CreditScore        10000 non-null int64
Geography          10000 non-null object
Gender             10000 non-null object
Age                10000 non-null int64
Tenure             10000 non-null int64
Balance            10000 non-null float64
NumOfProducts      10000 non-null int64
HasCrCard          10000 non-null int64
IsActiveMember     10000 non-null int64
EstimatedSalary    10000 non-null float64
Exited             10000 non-null int64
dtypes: float64(2), int64(7), object(2)
memory usage: 859.5+ KB


3. Distinguish the feature and target set (5 points)

In [0]:
# X = df['Exited']
# Y = df.drop(['Exited'],axis=1)
X = df.iloc[:, 0:10].values
y = df.iloc[:,10].values

In [11]:
X

array([[619, 'France', 'Female', ..., 1, 1, 101348.88],
       [608, 'Spain', 'Female', ..., 0, 1, 112542.58],
       [502, 'France', 'Female', ..., 1, 0, 113931.57],
       ...,
       [709, 'France', 'Female', ..., 0, 1, 42085.58],
       [772, 'Germany', 'Male', ..., 1, 0, 92888.52],
       [792, 'France', 'Female', ..., 1, 0, 38190.78]], dtype=object)

In [12]:
y

array([1, 0, 1, ..., 1, 1, 0])

In [0]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
labelencoder_X_1 = LabelEncoder()
X[:,1] = labelencoder_X_1.fit_transform(X[:,1])
labelencoder_X_2 = LabelEncoder()
X[:,2] = labelencoder_X_2.fit_transform(X[:,2])

In [14]:
onehotencoder = OneHotEncoder(categorical_features=[1])
X = onehotencoder.fit_transform(X).toarray()
X = X[:,1:]

/usr/local/lib/python3.6/dist-packages/sklearn/preprocessing/_encoders.py:415: FutureWarning: The handling of integer data will change in version 0.22. Currently, the categories are determined based on the range [0, max(values)], while in the future they will be determined based on the unique values.
If you want the future behaviour and silence this warning, you can specify "categories='auto'".
In case you used a LabelEncoder before this OneHotEncoder to convert the categories to integers, then you can now use the OneHotEncoder directly.
  warnings.warn(msg, FutureWarning)
/usr/local/lib/python3.6/dist-packages/sklearn/preprocessing/_encoders.py:451: DeprecationWarning: The 'categorical_features' keyword is deprecated in version 0.20 and will be removed in 0.22. You can use the ColumnTransformer instead.
  "use the ColumnTransformer instead.", DeprecationWarning)


In [0]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
labelencoder_X_1 = LabelEncoder()
X[:,1] = labelencoder_X_1.fit_transform(X[:,1])
labelencoder_X_2 = LabelEncoder()
X[:,2] = labelencoder_X_2.fit_transform(X[:,2])

In [16]:
onehotencoder = OneHotEncoder(categorical_features=[1])
X = onehotencoder.fit_transform(X).toarray()
X = X[:,1:]

/usr/local/lib/python3.6/dist-packages/sklearn/preprocessing/_encoders.py:415: FutureWarning: The handling of integer data will change in version 0.22. Currently, the categories are determined based on the range [0, max(values)], while in the future they will be determined based on the unique values.
If you want the future behaviour and silence this warning, you can specify "categories='auto'".
In case you used a LabelEncoder before this OneHotEncoder to convert the categories to integers, then you can now use the OneHotEncoder directly.
  warnings.warn(msg, FutureWarning)
/usr/local/lib/python3.6/dist-packages/sklearn/preprocessing/_encoders.py:451: DeprecationWarning: The 'categorical_features' keyword is deprecated in version 0.20 and will be removed in 0.22. You can use the ColumnTransformer instead.
  "use the ColumnTransformer instead.", DeprecationWarning)


4. Divide the data set into Train and test sets

In [17]:
xtrain,xtest,ytrain,ytest = train_test_split(X,y,test_size=.3,random_state=4)
xtrain.shape

(7000, 11)

In [18]:
ytrain.shape

(7000,)

5. Normalize the train and test data (5 points)

In [0]:
sc = StandardScaler()
xtrain = sc.fit_transform(xtrain)
xtest = sc.transform(xtest)

6. Initialize & build the model (10 points)

In [20]:
# Building a Model
classifier = Sequential()
classifier.add(Dense(6, kernel_initializer = 'uniform', activation = 'relu'))
classifier.add(Dense(6, kernel_initializer = 'uniform', activation = 'relu'))
classifier.add(Dense(1, kernel_initializer = 'uniform', activation = 'sigmoid'))

W0901 07:47:20.816228 139906429024128 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:66: The name tf.get_default_graph is deprecated. Please use tf.compat.v1.get_default_graph instead.



In [21]:
# Compiling the Model
classifier.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

W0901 07:47:20.860563 139906429024128 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/optimizers.py:793: The name tf.train.Optimizer is deprecated. Please use tf.compat.v1.train.Optimizer instead.



In [22]:
classifier.fit(xtrain, ytrain, batch_size = 10, epochs = 100)

W0901 07:47:20.895152 139906429024128 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:541: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.

W0901 07:47:20.899654 139906429024128 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:4432: The name tf.random_uniform is deprecated. Please use tf.random.uniform instead.

W0901 07:47:20.950951 139906429024128 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:3657: The name tf.log is deprecated. Please use tf.math.log instead.

W0901 07:47:20.957911 139906429024128 deprecation.py:323] From /usr/local/lib/python3.6/dist-packages/tensorflow/python/ops/nn_impl.py:180: add_dispatch_support.<locals>.wrapper (from tensorflow.python.ops.array_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use tf.where in 

Epoch 1/100
7000/7000 [==============================] - 1s 197us/step - loss: 0.4966 - acc: 0.7926
Epoch 2/100
7000/7000 [==============================] - 1s 111us/step - loss: 0.4340 - acc: 0.7924
Epoch 3/100
7000/7000 [==============================] - 1s 106us/step - loss: 0.4291 - acc: 0.7924
Epoch 4/100
7000/7000 [==============================] - 1s 107us/step - loss: 0.4243 - acc: 0.8067
Epoch 5/100
7000/7000 [==============================] - 1s 108us/step - loss: 0.4214 - acc: 0.8226
Epoch 6/100
7000/7000 [==============================] - 1s 105us/step - loss: 0.4198 - acc: 0.8256
Epoch 7/100
7000/7000 [==============================] - 1s 110us/step - loss: 0.4178 - acc: 0.8271
Epoch 8/100
7000/7000 [==============================] - 1s 112us/step - loss: 0.4157 - acc: 0.8289
Epoch 9/100
7000/7000 [==============================] - 1s 111us/step - loss: 0.4151 - acc: 0.8297
Epoch 10/100
7000/7000 [==============================] - 1s 107us/step - loss: 0.4140 - acc: 0.8314

7. Optimize the model (Optional)

8. Predict the results using 0.5 as a threshold (Optional)

In [23]:
y_pred = classifier.predict(xtest)
y_pred = (y_pred > 0.5)
y_pred

array([[False],
       [False],
       [False],
       ...,
       [False],
       [False],
       [False]])

Print the Accuracy score and confusion matrix (5 points)

In [24]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(ytest, y_pred)
cm

array([[2335,   81],
       [ 385,  199]])

In [25]:
accuracy = (cm[0][0]+cm[1][1])/(cm[0][0]+cm[0][1]+cm[1][0]+cm[1][1])
accuracy

0.8446666666666667

7. Optimize the model (Optional)

In [0]:
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout

In [0]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)

In [0]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [29]:
classifier = Sequential()

classifier.add(Dense(units = 6, kernel_initializer = 'uniform', activation = 'relu', input_dim = 11))
classifier.add(Dropout(rate = 0.1))

classifier.add(Dense(units = 6, kernel_initializer = 'uniform', activation = 'relu'))
classifier.add(Dropout(rate = 0.1))

classifier.add(Dense(units = 1, kernel_initializer = 'uniform', activation = 'sigmoid'))
classifier.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])
classifier.fit(X_train, y_train, batch_size = 10, epochs = 100)

W0901 07:48:39.233930 139906429024128 deprecation.py:506] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:3733: calling dropout (from tensorflow.python.ops.nn_ops) with keep_prob is deprecated and will be removed in a future version.
Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


Epoch 1/100
8000/8000 [==============================] - 1s 156us/step - loss: 0.5016 - acc: 0.7964
Epoch 2/100
8000/8000 [==============================] - 1s 116us/step - loss: 0.4401 - acc: 0.7967
Epoch 3/100
8000/8000 [==============================] - 1s 121us/step - loss: 0.4371 - acc: 0.7967
Epoch 4/100
8000/8000 [==============================] - 1s 120us/step - loss: 0.4375 - acc: 0.7967
Epoch 5/100
8000/8000 [==============================] - 1s 115us/step - loss: 0.4343 - acc: 0.7967
Epoch 6/100
8000/8000 [==============================] - 1s 117us/step - loss: 0.4342 - acc: 0.7967
Epoch 7/100
8000/8000 [==============================] - 1s 118us/step - loss: 0.4296 - acc: 0.7967
Epoch 8/100
8000/8000 [==============================] - 1s 121us/step - loss: 0.4314 - acc: 0.7967
Epoch 9/100
8000/8000 [==============================] - 1s 118us/step - loss: 0.4357 - acc: 0.7967
Epoch 10/100
8000/8000 [==============================] - 1s 119us/step - loss: 0.4297 - acc: 0.7967

In [30]:
classifier.predict(X_test)

array([[0.06933033],
       [0.10784021],
       [0.086557  ],
       ...,
       [0.13473102],
       [0.15911165],
       [0.30136502]], dtype=float32)

In [0]:
y_predict=classifier.predict_classes(X_test)

In [32]:
metrics.accuracy_score(y_test,y_predict)

0.832

In [33]:
metrics.confusion_matrix(y_test,y_predict)

array([[1563,   26],
       [ 310,  101]])

In [34]:
cr=metrics.classification_report(y_test,y_predict)
print(cr)

              precision    recall  f1-score   support

           0       0.83      0.98      0.90      1589
           1       0.80      0.25      0.38       411

    accuracy                           0.83      2000
   macro avg       0.81      0.61      0.64      2000
weighted avg       0.83      0.83      0.79      2000

